# Notebook 2: 2D Poiseuille Flow in FEniCS

## Objective
Solve the Stokes equations in a 2D channel driven by a pressure gradient.

**Problem:**
- Channel: $[0, L] \times [-H/2, H/2]$, $L=1$, $H=0.1$
- Pressure gradient: $\Delta p / L$ in $x$-direction (implemented as body force)
- No-slip walls: $\mathbf{u} = 0$ at $y = \pm H/2$
- Viscosity: $\mu = 1$

**Analytical solution (Hagen-Poiseuille):**
$$u_x(y) = \frac{\Delta p}{2 \mu L} \left( \frac{H^2}{4} - y^2 \right)$$

In [ ]:
from fenics import *
import numpy as np
import matplotlib.pyplot as plt

# Inline plots
%matplotlib inline

# Suppress FEniCS solver output
set_log_level(LogLevel.WARNING)

---
## Step 1: Mesh

In [ ]:
L  = 1.0   # Channel length
H  = 0.1   # Channel height
mu = 1.0   # Dynamic viscosity
dp = 1.0   # Pressure drop (p_inlet - p_outlet)

nx, ny = 40, 10
mesh = RectangleMesh(Point(0.0, -H/2), Point(L, H/2), nx, ny)

print(f"Mesh: {mesh.num_cells()} cells, {mesh.num_vertices()} vertices")

---
## Step 2: Mixed Function Space (P2-P1)

The correct way to define mixed spaces in FEniCS legacy is via `MixedElement`.

- **Velocity:** P2 (quadratic) — captures parabolic profile exactly
- **Pressure:** P1 (linear) — satisfies inf-sup (LBB) stability condition

In [ ]:
# Define elements
P2 = VectorElement("P", mesh.ufl_cell(), 2)  # Velocity: vector P2
P1 = FiniteElement("P", mesh.ufl_cell(), 1)  # Pressure: scalar P1

# Mixed element and space
TH = MixedElement([P2, P1])                  # Taylor-Hood element
W  = FunctionSpace(mesh, TH)                 # Mixed function space

print(f"Mixed space DOFs: {W.dim()}")
print(f"  Velocity sub-space DOFs: {W.sub(0).dim()}")
print(f"  Pressure sub-space DOFs: {W.sub(1).dim()}")

---
## Step 3: Boundary Conditions

In [ ]:
tol = 1e-10

def top_wall(x, on_boundary):
    return on_boundary and abs(x[1] - H/2) < tol

def bottom_wall(x, on_boundary):
    return on_boundary and abs(x[1] + H/2) < tol

# No-slip: u = (0, 0) on top and bottom walls
noslip = Constant((0.0, 0.0))
bc_top    = DirichletBC(W.sub(0), noslip, top_wall)
bc_bottom = DirichletBC(W.sub(0), noslip, bottom_wall)
bcs = [bc_top, bc_bottom]

print("No-slip BCs applied to top and bottom walls.")

---
## Step 4: Weak Form

The Stokes weak form is: find $(\mathbf{u}, p) \in V \times Q$ such that

$$\mu \int \nabla \mathbf{u} : \nabla \mathbf{v} \, dx - \int p \, \nabla \cdot \mathbf{v} \, dx + \int q \, \nabla \cdot \mathbf{u} \, dx = \int \mathbf{f} \cdot \mathbf{v} \, dx$$

The pressure gradient $\Delta p / L$ is applied as a body force $\mathbf{f} = (-\Delta p / L, 0)$.

In [ ]:
# Trial and test functions from the mixed space
(u, p) = TrialFunctions(W)
(v, q) = TestFunctions(W)

# Body force = pressure gradient (drives the flow)
f = Constant((-dp/L, 0.0))

# Bilinear form (Stokes)
a = (mu * inner(grad(u), grad(v))
     - p * div(v)
     + q * div(u)) * dx

# Linear form (body force)
L_form = inner(f, v) * dx

print("Weak form defined.")

---
## Step 5: Solve

In [ ]:
w = Function(W)
solve(a == L_form, w, bcs)

# Split velocity and pressure
u_sol, p_sol = w.split()

print("Stokes solved.")

---
## Step 6: Validate Against Analytical Solution

In [ ]:
def u_analytical(y):
    """Hagen-Poiseuille: parabolic velocity profile."""
    return (dp / (2.0 * mu * L)) * (H**2 / 4.0 - y**2)

# Sample along vertical line at x = L/2
y_pts = np.linspace(-H/2 + 1e-6, H/2 - 1e-6, 60)
x_mid = L / 2.0

u_fem = np.array([u_sol(x_mid, y)[0] for y in y_pts])
u_ana = np.array([u_analytical(y) for y in y_pts])

L2_err = np.sqrt(np.trapz((u_fem - u_ana)**2, y_pts))
print(f"Max u_x  (analytical): {u_ana.max():.6f}")
print(f"Max u_x  (FEM):        {u_fem.max():.6f}")
print(f"L2 error along centre: {L2_err:.2e}")

---
## Step 7: Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- velocity profile ---
ax = axes[0]
ax.plot(u_ana, y_pts, 'b-',  lw=2.5, label='Analytical')
ax.plot(u_fem, y_pts, 'ro', ms=4,   label='FEM')
ax.set_xlabel('$u_x$', fontsize=12)
ax.set_ylabel('$y$',   fontsize=12)
ax.set_title('Velocity profile at $x=L/2$', fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)

# --- velocity magnitude (colour map) ---
ax = axes[1]
V_scalar = FunctionSpace(mesh, "P", 1)
u_mag = project(sqrt(u_sol[0]**2 + u_sol[1]**2), V_scalar)
c1 = plot(u_mag, ax=ax, cmap='viridis')
plt.colorbar(c1, ax=ax, label='$|\mathbf{u}|$')
ax.set_title('Velocity magnitude', fontsize=12)

# --- pressure ---
ax = axes[2]
c2 = plot(p_sol, ax=ax, cmap='RdBu_r')
plt.colorbar(c2, ax=ax, label='$p$')
ax.set_title('Pressure', fontsize=12)

plt.tight_layout()
plt.savefig('/tmp/poiseuille.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8: Divergence-free Check

In [ ]:
div_u = project(div(u_sol), V_scalar)
div_max = div_u.vector().norm('linf')
print(f"max |∇·u| = {div_max:.2e}  (should be ≈ machine precision)")

---
## Summary

| Step | Key Point |
|------|-----------|
| Mixed element | `MixedElement([P2, P1])` — never `V * Q` in legacy FEniCS |
| Weak form | Saddle-point structure: viscous + pressure + continuity |
| Body force | Pressure gradient → uniform body force $f = -\Delta p / L$ |
| Validation | L2 error vs Hagen-Poiseuille; divergence ≈ 0 |

**Exercise:** Change `dp` or `H` and verify the maximum velocity scales as $u_{max} \propto \Delta p H^2$.